# maniol-elvish シーン別 Classifier 学習・評価（`scene-classifier-train-001`）

整理済み参照データ（`maniol-elvish_inspection001_for_learn.zip`）を使って、**骨格検出（`rtmlib`）を使わずに DINOv2 埋め込み + WD14-Tagger タグ特徴量でシーン別分類器を学習・評価**するノートブックです。

- **ベースシーン分類器（フラット分類）**: `<向き>/<フレーミング>/<ベースアングル>`（`broken_anatomy` / `lower_body` / `mid_body` は除外）
- **修飾フラグ分類器（独立 2 値分類）**: `is_sitting` / `is_twisted` を独立判定し、推論時に `<base_angle>__sitting` / `<base_angle>__twisted` へ合成
- **出力先**: Google Drive の `Colab Notebooks/snapshot-inspector/maniol-elvish/inspection_002/model/`（特徴量キャッシュ `.npz` + 学習済みモデル `.joblib` + `metadata.json`）

1. **セットアップ**: 必要パッケージ導入と GPU 確認（`rtmlib` 不使用）
2. **Drive マウント & 共通モジュール・2 モデル初期化**: DINOv2 + WD14-Tagger
3. **学習データ展開 & ラベル集計**: ZIP 展開・除外クラス確認・ベースクラス分布確認
4. **特徴量抽出 & Drive キャッシュ**: md5 ベースでキャッシュ（2 回目以降は数秒で読込）
5. **Stratified K-Fold CV によるモデル比較**: `DINOv2 のみ` vs `DINOv2 + WD14` × 複数分類器
6. **OOF 誤分類分析・混同ペア確認**: 境界ケースの目視確認と信頼度分布チェック
7. **本番モデル学習 & Drive 保存**: 全クラスで学習し `.joblib` + `metadata.json` を保存

## 1. セットアップ

In [ ]:
%pip install -q opencv-python huggingface_hub transformers pillow pandas matplotlib scikit-learn joblib
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir onnxruntime-gpu

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 共通モジュール・2 モデル初期化

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

# ローカル実行時は親ディレクトリを優先し、Colab 上ではリポジトリを clone/pull する
local_inspector_root = Path("..").resolve()
if (local_inspector_root / "common" / "__init__.py").exists():
    INSPECTOR_ROOT = local_inspector_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    INSPECTOR_ROOT = REPO_DIR / "colab" / "snapshot-inspector"

if str(INSPECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTOR_ROOT))

import os
import shutil
import zipfile
import pandas as pd

from common import (
    DEFAULT_DINOV2_REPO,
    DEFAULT_WD14_REPO,
    DINOv2Embedder,
    EXCLUDED_SCENE_FRAMINGS,
    WD14Tagger,
    discover_labeled_dataset,
    evaluate_scene_classifiers_v2,
    extract_or_load_scene_feature_pack,
    get_onnx_device_config,
    prepare_scene_dataset,
    save_results_to_drive,
    save_scene_classifier_bundle,
    train_scene_classifier_bundle_v2,
)

_, ort_providers = get_onnx_device_config()
print(f"INSPECTOR_ROOT: {INSPECTOR_ROOT}")
print(f"ONNX providers: {ort_providers}")

# 2 モデル初期化（DINOv2 + WD14-Tagger v3 / rtmlib 不使用）
DINOV2_REPO = DEFAULT_DINOV2_REPO  # 例: "facebook/dinov2-base"
WD14_REPO = DEFAULT_WD14_REPO      # 例: "SmilingWolf/wd-eva02-large-tagger-v3"

dinov2_embedder = DINOv2Embedder(model_id=DINOV2_REPO)
wd14_tagger = WD14Tagger(repo_id=WD14_REPO, providers=ort_providers)


## 3. 学習データ展開 & ラベル集計

In [ ]:
CHARACTER_NAME = "maniol-elvish"
DRIVE_BASE_DIR = f"/content/drive/MyDrive/Colab Notebooks/snapshot-inspector/{CHARACTER_NAME}/inspection_002"
TRAIN_ZIP_PATH = f"{DRIVE_BASE_DIR}/maniol-elvish_inspection001_for_learn.zip"
DRIVE_MODEL_DIR = f"{DRIVE_BASE_DIR}/model"
DRIVE_CACHE_DIR = f"{DRIVE_BASE_DIR}/cache"

LOCAL_TRAIN_ROOT = f"./data/{CHARACTER_NAME}/train_extracted"
LOCAL_WORK_DIR = f"./output/{CHARACTER_NAME}/train_001"
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

# ローカルに未展開なら ZIP を展開（高速読み込みのためローカル SSD へ展開する）
if not os.path.exists(LOCAL_TRAIN_ROOT) or len(os.listdir(LOCAL_TRAIN_ROOT)) == 0:
    os.makedirs(LOCAL_TRAIN_ROOT, exist_ok=True)
    print(f"Unzipping: {TRAIN_ZIP_PATH} -> {LOCAL_TRAIN_ROOT}")
    with zipfile.ZipFile(TRAIN_ZIP_PATH, "r") as zf:
        zf.extractall(LOCAL_TRAIN_ROOT)

# ルートに単一フォルダ（maniol-elvish_inspection001_for_learn/）が入っている場合はその配下を起点にする
subdirs = [p for p in Path(LOCAL_TRAIN_ROOT).iterdir() if p.is_dir()]
DATASET_DIR = str(subdirs[0]) if len(subdirs) == 1 else LOCAL_TRAIN_ROOT
print(f"DATASET_DIR: {DATASET_DIR}")

raw_df = discover_labeled_dataset(DATASET_DIR)
train_df, excluded_df = prepare_scene_dataset(raw_df, excluded_framings=EXCLUDED_SCENE_FRAMINGS)

print(f"\n総画像数: {len(raw_df)} 枚")
print(f"除外対象 ({sorted(EXCLUDED_SCENE_FRAMINGS)}): {len(excluded_df)} 枚")
if len(excluded_df) > 0:
    display(excluded_df["label_framing"].value_counts().to_frame("count"))

print(f"\n学習対象: {len(train_df)} 枚 | ベースシーン数: {train_df['base_scene'].nunique()}")
print(f"  is_sitting=1: {int(train_df['is_sitting'].sum())} 枚 ({train_df['is_sitting'].mean()*100:.1f}%)")
print(f"  is_twisted=1: {int(train_df['is_twisted'].sum())} 枚 ({train_df['is_twisted'].mean()*100:.1f}%)")
display(train_df["base_scene"].value_counts().to_frame("count").head(20))

## 4. 特徴量抽出 & Drive キャッシュ

DINOv2（768 次元）と WD14-Tagger 全 general タグ（float16）を抽出し、Drive 上の `inspection_002/cache/` に保存します。
2 回目以降は md5 一致により数秒で読み込みが完了します。

In [ ]:
FORCE_RECOMPUTE_FEATURES = False

# v2 特徴パック: DINOv2 空間パッチ特徴 (CLS + 左右差分 + 上下差分) + 水平フリップ特徴 + WD14
# 既存 v1 キャッシュがあれば WD14 スコアは自動流用されるため、DINOv2 のみ高速に再抽出されます
feature_pack = extract_or_load_scene_feature_pack(
    items_df=train_df,
    cache_dir=DRIVE_CACHE_DIR,
    embedder=dinov2_embedder,
    tagger=wd14_tagger,
    include_hflip=True,
    force_recompute=FORCE_RECOMPUTE_FEATURES,
    cache_prefix="train_scene_features",
)


## 5. Stratified K-Fold CV による特徴量・分類器比較

`dino_only` vs `dino_wd14`、および複数の軽量分類器候補を比較評価します。
サンプル数が `N_SPLITS` 未満の極小クラス（4 クラス）は全 fold の学習セットに含め、評価スコアは `>= N_SPLITS` クラスで算出します。

In [ ]:
N_SPLITS = 5
SPATIAL_WEIGHT = 0.7
WD14_MIN_MAX_SCORE = 0.15
WD14_MAX_TAGS = 512
WD14_WEIGHT = 1.0
USE_HFLIP_TTA = True

# 従来 (dino_wd14) vs 空間パッチ特徴 (dino_spatial_wd14) × 水平フリップ 2 倍学習 (False/True) を CV 比較
comparison_df, best_oof = evaluate_scene_classifiers_v2(
    train_df=train_df,
    feature_pack=feature_pack,
    feature_modes=["dino_wd14", "dino_spatial_wd14"],
    hflip_aug_modes=[False, True],
    model_names=["logreg_strong", "logreg_c30", "logreg_balanced", "linear_svc_cal"],
    use_hflip_tta=USE_HFLIP_TTA,
    n_splits=N_SPLITS,
    spatial_weight=SPATIAL_WEIGHT,
    wd14_min_max_score=WD14_MIN_MAX_SCORE,
    wd14_max_tags=WD14_MAX_TAGS,
    wd14_weight=WD14_WEIGHT,
)

print("\n=== CV 比較結果 (v2: 空間特徴 + 水平フリップ拡張 + TTA) ===")
display(comparison_df)
print("\n最良設定:", best_oof["config"])


## 6. OOF 誤分類分析・混同ペア・信頼度分布確認

In [ ]:
oof_df = best_oof["oof_df"]
print("=== よくある混同ペア Top 20 ===")
display(best_oof["confusion_pairs"].head(20))

# 信頼度しきい値ごとのカバー率と精度を確認（推論時の min_confidence 決定の参考）
thr_rows = []
for thr in [0.0, 0.30, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80]:
    sub = oof_df[oof_df["oof_prob"] >= thr]
    cov = len(sub) / len(oof_df)
    acc = sub["oof_correct"].mean() if len(sub) > 0 else 0.0
    thr_rows.append({
        "min_confidence": thr,
        "coverage": round(cov, 4),
        "kept_images": len(sub),
        "uncertain_images": len(oof_df) - len(sub),
        "accuracy_on_kept": round(float(acc), 4),
    })
print("\n=== min_confidence しきい値シミュレーション（OOF） ===")
display(pd.DataFrame(thr_rows))

## 7. 本番モデル学習 & Google Drive 保存

最良設定（または手動指定）で全学習データ（極小クラス含む全 59 ベースクラス + `sitting` / `twisted`）を学習し、Drive に保存します。

In [ ]:
# デフォルトは CV 最良設定を使用。手動固定したい場合はここを書き換えてください。
SELECTED_FEATURE_MODE = best_oof["config"]["feature_mode"]
SELECTED_MODEL_NAME = best_oof["config"]["model_name"]
SELECTED_HFLIP_AUG = best_oof["config"]["use_hflip_aug"]

bundle = train_scene_classifier_bundle_v2(
    train_df=train_df,
    feature_pack=feature_pack,
    feature_mode=SELECTED_FEATURE_MODE,
    model_name=SELECTED_MODEL_NAME,
    use_hflip_aug=SELECTED_HFLIP_AUG,
    spatial_weight=SPATIAL_WEIGHT,
    wd14_min_max_score=WD14_MIN_MAX_SCORE,
    wd14_max_tags=WD14_MAX_TAGS,
    wd14_weight=WD14_WEIGHT,
    dinov2_model_id=DINOV2_REPO,
    wd14_repo_id=WD14_REPO,
    cv_metrics=best_oof["metrics"],
)

saved = save_scene_classifier_bundle(bundle, DRIVE_MODEL_DIR)

# CV 比較表と OOF 結果も Drive へ保存
comp_csv = os.path.join(LOCAL_WORK_DIR, "cv_comparison.csv")
oof_csv = os.path.join(LOCAL_WORK_DIR, "oof_predictions.csv")
conf_csv = os.path.join(LOCAL_WORK_DIR, "oof_confusion_pairs.csv")
comparison_df.to_csv(comp_csv, index=False, encoding="utf-8-sig")
oof_df.to_csv(oof_csv, index=False, encoding="utf-8-sig")
best_oof["confusion_pairs"].to_csv(conf_csv, index=False, encoding="utf-8-sig")

save_results_to_drive(DRIVE_MODEL_DIR, file_paths=[comp_csv, oof_csv, conf_csv])


### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する